In [ ]:
import torch
import matplotlib.pyplot as plt
from model.resnet11 import MnistFM

## Config

In [ ]:
model_path = "/workspace/samples/sample_model/resnet11_epoch100.pth" # Load Model

device = torch.device("cuda" if torch.cuda.is_available() else "cpu")

## Load Model

In [ ]:
def load_eval_model(model_path, device="cuda"):
    model = MnistFM()
    model = model.to(device)

    checkpoint_path = model_path
    checkpoint = torch.load(checkpoint_path, map_location=device)

    model.load_state_dict(checkpoint["model_state_dict"])
    model.eval()
    return model

model = load_eval_model(model_path, device)

## Inference Preparation

In [ ]:
@torch.no_grad()
def generate(model, digit, device, steps, check_interval=10):
    model.eval()

    pseudo_prompt = torch.tensor([digit], device=device)

    x = model.sample_noise((1, 1, 28, 28), device)

    trajectory = [x.detach().cpu().clone()]

    dt = 1.0 / steps
    for i in range(steps):
        t = torch.tensor([i / steps], device=device, dtype=torch.float32)

        v = model(pseudo_prompt, t, x)

        x = x + v * dt

        if (i + 1) % check_interval == 0:
            trajectory.append(x.detach().cpu().clone())

    return trajectory

def plot_trajectory(trajectory, digit, check_interval):
    """
    trajectory: generate_with_trajectory() の返り値
    """
    num_images = len(trajectory)

    plt.figure(figsize=(2 * num_images, 2.5))

    for i, x in enumerate(trajectory):
        img = x[0, 0].numpy() 

        plt.subplot(1, num_images, i + 1)
        plt.imshow(img, cmap="gray", vmin=-1, vmax=1)
        plt.title(f"step {i*check_interval}")
        plt.axis("off")

    plt.suptitle(f"Generation trajectory for digit={digit}", fontsize=14)
    plt.tight_layout()
    plt.show()

## Inference

度々適切な画像を生成できません。<br>
初期ノイズが適切に学習できてないベクトル場に乗った場合にそうなると考えられます。

In [ ]:
digit = 3 # 生成させたい数字. 0~9で入力
steps = 10 # デノイズステップ数
check_interval = 1
device = torch.device("cuda" if torch.cuda.is_available() else "cpu")
trajectory = generate(model, digit=digit, device=device, steps=steps, check_interval=check_interval)
plot_trajectory(trajectory, digit=digit, check_interval=check_interval)